In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# I-monitor o i-cancel ang isang job

Tingnan ang listahan ng iyong mga workload sa [Workloads page](https://quantum.cloud.ibm.com/workloads).

## Tingnan ang status ng job
Pumunta sa iyong [Workloads table](https://quantum.cloud.ibm.com/workloads) at tingnan ang Status column para malaman kung natapos na o nabigo ang isang job.

## Tingnan ang natitirang paggamit
Pumunta sa iyong [Instances table](https://quantum.cloud.ibm.com/instances) at piliin ang tab na nauugnay sa plano na gusto mong tingnan ang natitirang paggamit. Ipinapakita ang kabuuang oras na ginamit at kabuuang oras na natitira sa iyong plano.

## Tingnan ang mga sukatan sa bilang ng mga job at workload na na-submit
Pumunta sa [Analytics page](https://quantum.cloud.ibm.com/analytics) para makita ang kabuuang bilang ng mga job na na-submit, kasama na ang bilang ng mga batch workload at session workload. Tandaan na makikita mo lang ang Analytics page para sa mga account na pag-aari mo o pinamamahalaan mo.

## I-monitor ang isang job
Gamitin ang job instance para suriin ang status ng job o kunin ang mga resulta sa pamamagitan ng pagtawag ng naaangkop na command:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Tingnan ang mga resulta ng job kaagad pagkatapos nitong matapos. Available ang mga resulta ng job pagkatapos matapos ang job. Kaya naman, ang job.result() ay isang blocking call hanggang sa matapos ang job. |
| job.job\_id()                 | Ibalik ang ID na natatanging nagtatukoy sa job na iyon. Kailangan ang job ID para makuha ang mga resulta ng job sa ibang pagkakataon. Kaya naman, inirerekomenda na i-save mo ang mga ID ng mga job na maaaring gusto mong kunin sa ibang pagkakataon. |
| job.status()                  | Suriin ang status ng job.                                                                                                                                                                                    |
| job = service.job(\<job\_id>) | Kunin ang isang job na na-submit mo noon. Kailangan ng job ID para sa call na ito.                                                                                                                           |

<span id="retrieve-later"></span>
## Kunin ang mga resulta ng job sa ibang pagkakataon
Tumawag ng `service.job(\<job\_id>)` para makuha ang isang job na na-submit mo noon. Kung wala kang job ID, o kung gusto mong makuha ang maraming job nang sabay-sabay — kasama na ang mga job mula sa mga retired na QPU (quantum processing units) — tumawag na lang ng `service.jobs()` na may optional na mga filter. Tingnan ang [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** Ibinabalik din ng `service.jobs()` ang mga job na pinatakbo mula sa deprecated na `qiskit-ibm-provider` package. Hindi na available ang mga job na na-submit ng mas lumang (at deprecated na rin) `qiskit-ibmq-provider` package.

### Halimbawa
Ibinabalik ng halimbawang ito ang 10 pinakabagong runtime job na pinatakbo sa `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## I-cancel ang isang job
Maaari mong i-cancel ang isang job mula sa IBM Quantum Platform dashboard sa pamamagitan ng Workloads page o ng details page para sa isang partikular na workload. Sa Workloads page, i-click ang overflow menu sa dulo ng row para sa workload na iyon, at piliin ang Cancel. Kung nasa details page ka para sa isang partikular na workload, gamitin ang Actions dropdown sa itaas ng pahina, at piliin ang Cancel.

Sa Qiskit, gamitin ang `job.cancel()` para i-cancel ang isang job.

<span id="execution-spans"></span>
## Tingnan ang mga execution span ng Sampler
Ang mga resulta ng mga [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) job na pinapatakbo sa Qiskit Runtime ay naglalaman ng impormasyon sa timing ng pagpapatupad sa kanilang metadata.
Magagamit ang timing information na ito para maglagay ng upper at lower timestamp bound sa kung kailan partikular na mga shot ang pinatupad sa QPU.
Ang mga shot ay pinagsama-sama sa mga [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span) object, na bawat isa ay nagpapakita ng oras ng pagsisimula, oras ng pagtatapos, at detalye kung aling mga shot ang nakolekta sa span.

Tinutukoy ng execution span kung aling data ang pinatupad sa loob ng window nito sa pamamagitan ng pagbibigay ng [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. Ang method na ito, kapag binigyan ng anumang [Primitive Unified Block (PUB)](/guides/primitive-input-output) index, ay nagbabalik ng boolean mask na `True` para sa lahat ng shot na pinatupad sa loob ng window nito. Ang mga PUB ay ina-index ayon sa pagkakasunod-sunod na ibinigay ang mga ito sa Sampler run call. Halimbawa, kung ang isang PUB ay may shape na `(2, 3)` at pinatakbo na may apat na shot, ang shape ng mask ay `(2, 3, 4)`. Tingnan ang [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) API page para sa buong detalye.

Halimbawa:
Para makita ang impormasyon ng execution span, suriin ang metadata ng resulta na ibinalik ng `SamplerV2`, na nasa anyo ng `ExecutionSpans` object. Ang object na ito ay isang list-like container na naglalaman ng mga instance ng mga subclass ng `ExecutionSpan`, tulad ng `SliceSpan`: